In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
df = pd.read_csv("../../data/egx/COMI.csv")
df.head()

In [ ]:
df["date"] = pd.to_datetime(df["date"])

In [ ]:
df["MA9"] = df["close"].rolling(window=9).mean()

df["MA20"] = df["close"].rolling(window=20).mean()

df.head(25)

In [ ]:
plt.figure(figsize=(15,6))

plt.plot(df["date"], df["close"], label="Close")

plt.plot(df["date"], df["MA9"], label="MA9")

plt.plot(df["date"], df["MA20"], label="MA20")

plt.legend()

plt.grid(True)

plt.show()

In [ ]:
cash = 1000
shares = 0

buy_count = 0
sell_count = 0

portfolio_values = []

buy_dates = []
buy_prices = []

sell_dates = []
sell_prices = []

In [ ]:
portfolio_values = [cash]

In [ ]:
for i in range(1, len(df)):

    price = df.loc[i, "close"]

    ma9 = df.loc[i, "MA9"]
    ma20 = df.loc[i, "MA20"]

    prev_ma9 = df.loc[i-1, "MA9"]
    prev_ma20 = df.loc[i-1, "MA20"]

    if pd.isna(ma9) or pd.isna(ma20):
        portfolio_values.append(cash + shares * price)
        continue

    # Buy Crossover
    if prev_ma9 <= prev_ma20 and ma9 > ma20 and shares == 0:

        shares = cash / price
        cash = 0

        buy_count += 1

        buy_dates.append(df.loc[i, "date"])
        buy_prices.append(price)

    # Sell Crossover
    elif prev_ma9 >= prev_ma20 and ma9 < ma20 and shares > 0:

        cash = shares * price
        shares = 0

        sell_count += 1

        sell_dates.append(df.loc[i, "date"])
        sell_prices.append(price)

    portfolio_values.append(cash + shares * price)

In [ ]:
print(len(df))
print(len(portfolio_values))

In [ ]:
df["Portfolio"] = portfolio_values

In [ ]:
print("Final Portfolio Value:", round(df["Portfolio"].iloc[-1], 2))

In [ ]:
print("Buy Operations :", buy_count)
print("Sell Operations:", sell_count)

In [ ]:
running_max = df["Portfolio"].cummax()

drawdown = (running_max - df["Portfolio"]) / running_max

max_drawdown = drawdown.max()

print("Maximum Drawdown:", round(max_drawdown * 100, 2), "%")

In [ ]:
plt.figure(figsize=(16,8))

plt.plot(df["date"], df["close"], label="Close", alpha=0.6)
plt.plot(df["date"], df["MA9"], label="MA9")
plt.plot(df["date"], df["MA20"], label="MA20")

plt.scatter(
    buy_dates,
    buy_prices,
    marker="^",
    s=100,
    label="Buy"
)

plt.scatter(
    sell_dates,
    sell_prices,
    marker="v",
    s=100,
    label="Sell"
)

plt.title("Moving Average Crossover Strategy")
plt.xlabel("Date")
plt.ylabel("Price")

plt.legend()

plt.grid(True)

plt.show()

In [ ]:
df["Portfolio"] = portfolio_values

print("Final Portfolio Value:", round(df["Portfolio"].iloc[-1], 2))

running_max = df["Portfolio"].cummax()
drawdown = (running_max - df["Portfolio"]) / running_max
max_drawdown = drawdown.max()

print("Maximum Drawdown:", round(max_drawdown * 100, 2), "%")